In [2]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

script_dir = os.getcwd() 

FINAL_COMP_DIR = os.path.join(script_dir, 'final_comparison')
os.makedirs(FINAL_COMP_DIR, exist_ok=True)
print(f"📁 Folder output untuk CSV disiapkan di: {FINAL_COMP_DIR}\n")

subjects = [f"SUB{i}" for i in range(1, 13)]

# PERBAIKAN: Menambahkan 'WER' tepat setelah 'cer' pada daftar metrik
metrics = ['cer', 'WER', 'BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'ROUGE-1-P', 'ROUGE-1-R', 'ROUGE-1-F', 'BertScore']

print("Mengekstrak dan menghitung nilai seluruh metrik dari eksperimen...\n")

experiments = [
    {
        'Decoder': 'LSTM',
        'Train Data': 'Single',
        'type': 'single',
        'template': "complete_metrics_{subject}_eq_3_0_fixed_hilbert_test_predictions_6_1.csv"
    },
    {
        'Decoder': 'IndoGPT',
        'Train Data': 'Single',
        'type': 'single',
        'template': "complete_metrics_{subject}_eq_3_0_hilbert_test_predictions_10_1_IndoGPT.csv"
    },
    {
        'Decoder': 'LSTM',
        'Train Data': 'All',
        'type': 'all',
        'template': "complete_metrics_all_eq_3_0_fixed_hilbert_test_predictions_1_1.csv"
    },
    {
        'Decoder': 'IndoGPT',
        'Train Data': 'All',
        'type': 'all',
        'template': "complete_metrics_all_eq_3_0_hilbert_test_predictions_2_1_IndoGPT.csv"
    }
]

tables_data = {m: [] for m in metrics}

summary_rows = []


for exp in experiments:
    summary_row = {
        'Decoder': exp['Decoder'],
        'Train Data': exp['Train Data']
    }
    
    exp_metric_rows = {m: {'Decoder': exp['Decoder'], 'Train Data': exp['Train Data']} for m in metrics}
    
    raw_values = {m: [] for m in metrics}
    
    if exp['type'] == 'single':
        for subject in subjects:
            filename = exp['template'].format(subject=subject)
            filepath = os.path.join(script_dir, filename)
            
            if os.path.exists(filepath):
                try:
                    df = pd.read_csv(filepath)
                    for m in metrics:
                        if m in df.columns:
                            exp_metric_rows[m][subject] = round(df[m].mean(), 4)
                            raw_values[m].extend(df[m].dropna().tolist())
                        else:
                            exp_metric_rows[m][subject] = np.nan
                except Exception as e:
                    print(f"  [ERROR] Gagal membaca {filename}: {e}")
                    for m in metrics: exp_metric_rows[m][subject] = np.nan
            else:
                for m in metrics: exp_metric_rows[m][subject] = np.nan
                
    elif exp['type'] == 'all':
        filename = exp['template']
        filepath = os.path.join(script_dir, filename)
        
        if os.path.exists(filepath):
            try:
                df_all = pd.read_csv(filepath)
                for subject in subjects:
                    sub_df = df_all[df_all['subject'] == subject]
                    if len(sub_df) > 0:
                        for m in metrics:
                            if m in sub_df.columns:
                                exp_metric_rows[m][subject] = round(sub_df[m].mean(), 4)
                                raw_values[m].extend(sub_df[m].dropna().tolist())
                            else:
                                exp_metric_rows[m][subject] = np.nan
                    else:
                        for m in metrics: exp_metric_rows[m][subject] = np.nan
            except Exception as e:
                print(f"  [ERROR] Gagal membaca {filename}: {e}")
                for subject in subjects:
                    for m in metrics: exp_metric_rows[m][subject] = np.nan
        else:
            print(f"  ⚠️ [INFO] File {filename} belum ditemukan.")
            for subject in subjects:
                for m in metrics: exp_metric_rows[m][subject] = np.nan

    for m in metrics:
        if raw_values[m]:
            global_avg = round(np.mean(raw_values[m]), 4)
        else:
            global_avg = np.nan
            
        exp_metric_rows[m]['Rata-rata Global'] = global_avg
        tables_data[m].append(exp_metric_rows[m])
        
        summary_row[m] = global_avg
        
    summary_rows.append(summary_row)

column_order = ['Decoder', 'Train Data'] + subjects + ['Rata-rata Global']

for m in metrics:
    df_metric = pd.DataFrame(tables_data[m])
    df_metric = df_metric[[col for col in column_order if col in df_metric.columns]]
    
    print("=" * 110)
    print(f"TABEL PERBANDINGAN PERFORMA: {m.upper()}")
    print("=" * 110)
    display(df_metric)
    
    metric_csv_path = os.path.join(FINAL_COMP_DIR, f"hilbert_normal_{m}.csv")
    df_metric.to_csv(metric_csv_path, index=False)
    print(f"✅ Tersimpan: {metric_csv_path}\n")

df_summary_global = pd.DataFrame(summary_rows)

summary_col_order = ['Decoder', 'Train Data'] + metrics
df_summary_global = df_summary_global[[col for col in summary_col_order if col in df_summary_global.columns]]

print("*" * 110)
print("🏆 TABEL GRAND SUMMARY: RATA-RATA GLOBAL SELURUH METRIK 🏆")
print("*" * 110)
display(df_summary_global)

summary_csv_path = os.path.join(FINAL_COMP_DIR, "hilbert_normal_grand_summary.csv")
df_summary_global.to_csv(summary_csv_path, index=False)
print(f"✅ Tersimpan: {summary_csv_path}\n")


CURRENT_DIR = os.getcwd()
DATASET_DIR = os.path.abspath(os.path.join(CURRENT_DIR, '../../../dataset'))

oov_summary_list = []

print("\n\n" + "=" * 110)
print("MEMULAI ANALISIS OOV KARAKTER...")
print("=" * 110)

for subject in subjects:
    train_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_train.csv")
    val_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_val.csv")
    test_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_test.csv")
    
    if os.path.exists(train_file) and os.path.exists(val_file) and os.path.exists(test_file):
        df_train = pd.read_csv(train_file)
        df_val = pd.read_csv(val_file)
        df_test = pd.read_csv(test_file)
        
        # Gunakan astype(str) untuk mencegah error jika ada data null/kosong
        train_text = "".join(df_train['sentence'].astype(str).tolist())
        test_text = "".join(df_test['sentence'].astype(str).tolist())
        
        train_chars = set(train_text)
        test_chars = set(test_text)
        
        oov_chars = test_chars - train_chars
        
        test_text_len = len(test_text)
        if test_text_len > 0:
            oov_occurrences = sum(1 for char in test_text if char in oov_chars)
            oov_rate = (oov_occurrences / test_text_len) * 100
        else:
            oov_occurrences = 0
            oov_rate = 0.0

        oov_summary_list.append({
            'Subject': subject,
            'Train (Baris)': len(df_train),
            'Val (Baris)': len(df_val),
            'Test (Baris)': len(df_test),
            'Kamus Train': len(train_chars),
            'Kamus Test': len(test_chars),
            'Karakter OOV': "".join(sorted(list(oov_chars))) if oov_chars else "-",
            'Total Muncul OOV': oov_occurrences,
            'OOV Rate (%)': round(oov_rate, 4)
        })
    else:
        # Jika salah satu dari 3 file per subjek tidak ada, berikan peringatan
        print(f"  ⚠️ Melewati {subject}: File train/val/test tidak lengkap di folder dataset.")

if oov_summary_list:
    df_summary = pd.DataFrame(oov_summary_list)

    print("\n" + "=" * 110)
    print("RINGKASAN DATA SPLIT & OOV KARAKTER PER SUBJEK")
    print("=" * 110)
    display(df_summary)
    
    oov_csv_path = os.path.join(FINAL_COMP_DIR, "oov_tokenizer_char.csv")
    df_summary.to_csv(oov_csv_path, index=False)
    print(f"✅ Tersimpan: {oov_csv_path}\n")
else:
    print("❌ Tidak ada data yang bisa ditampilkan. Pastikan file *_eq_3_0_*.csv sudah dibuat di dalam folder /dataset.")

📁 Folder output untuk CSV disiapkan di: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison

Mengekstrak dan menghitung nilai seluruh metrik dari eksperimen...

TABEL PERBANDINGAN PERFORMA: CER


,Decoder,Train Data,SUB1,SUB2,SUB3,SUB4,SUB5,SUB6,SUB7,SUB8,SUB9,SUB10,SUB11,SUB12,Rata-rata Global
0,LSTM,Single,73.1763,72.7230,72.4228,76.3911,76.1458,75.5610,76.6472,75.9258,75.2913,73.5564,80.7038,83.4326,75.6653
1,IndoGPT,Single,74.4871,80.0865,94.5446,80.4264,81.0386,78.4388,77.3508,74.0824,75.9028,74.8622,75.2092,73.9088,77.7309
2,LSTM,All,74.8834,73.4683,75.2256,74.8789,73.1234,74.1187,73.1098,73.4189,77.7201,73.6300,74.3684,79.4974,74.5200
3,IndoGPT,All,72.6264,69.7910,78.5516,77.5785,72.4055,76.0558,75.0886,72.3638,76.1184,72.2043,70.8275,73.6028,74.0654


✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\hilbert_normal_cer.csv

TABEL PERBANDINGAN PERFORMA: WER


,Decoder,Train Data,SUB1,SUB2,SUB3,SUB4,SUB5,SUB6,SUB7,SUB8,SUB9,SUB10,SUB11,SUB12,Rata-rata Global
0,LSTM,Single,96.6667,100.0,100.0000,98.5833,100.0000,99.2500,101.5000,98.7500,98.9474,99.1228,101.9608,108.3333,99.6032
1,IndoGPT,Single,100.9167,100.0,111.6667,107.9167,100.0000,98.8333,95.8750,94.9405,97.4123,95.0877,100.0000,100.0000,99.7027
2,LSTM,All,99.4048,97.5,100.0000,98.0000,96.3333,96.9167,96.3333,97.0833,99.2105,96.8797,98.8235,106.2500,98.0096
3,IndoGPT,All,99.8214,100.0,103.3333,108.5417,100.2222,100.6667,103.3750,97.3333,109.6491,100.8772,97.8431,108.3333,102.2592


✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\hilbert_normal_WER.csv

TABEL PERBANDINGAN PERFORMA: BLEU-1


...

✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\hilbert_normal_BertScore.csv

**************************************************************************************************************
🏆 TABEL GRAND SUMMARY: RATA-RATA GLOBAL SELURUH METRIK 🏆
**************************************************************************************************************


,Decoder,Train Data,cer,WER,BLEU-1,BLEU-2,BLEU-3,BLEU-4,ROUGE-1-P,ROUGE-1-R,ROUGE-1-F,BertScore
0,LSTM,Single,75.6653,99.6032,2.2763,0.9113,0.7806,0.7362,4.3122,2.6946,3.2255,43.9982
1,IndoGPT,Single,77.7309,99.7027,2.9689,1.3095,1.0158,0.9154,6.8871,3.7377,4.6609,47.7373
2,LSTM,All,74.5200,98.0096,4.5037,1.7120,1.5939,1.4926,8.0423,5.1751,6.2127,51.9758
3,IndoGPT,All,74.0654,102.2592,2.3744,0.8668,0.7287,0.7547,3.5626,2.6394,2.9815,48.5036


✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\hilbert_normal_grand_summary.csv



MEMULAI ANALISIS OOV KARAKTER...

RINGKASAN DATA SPLIT & OOV KARAKTER PER SUBJEK


,Subject,Train (Baris),Val (Baris),Test (Baris),Kamus Train,Kamus Test,Karakter OOV,Total Muncul OOV,OOV Rate (%)
0,SUB1,70,10,20,24,23,-,0,0.0000
1,SUB2,35,5,10,23,23,z,1,0.3125
2,SUB3,35,5,10,23,21,-,0,0.0000
3,SUB4,70,10,20,24,22,-,0,0.0000
4,SUB5,35,5,10,23,23,-,0,0.0000
5,SUB6,70,10,20,23,22,-,0,0.0000
6,SUB7,70,10,20,24,22,-,0,0.0000
7,SUB8,70,10,20,23,23,-,0,0.0000
8,SUB9,64,9,19,24,22,-,0,0.0000
9,SUB10,63,9,19,23,23,-,0,0.0000


✅ Tersimpan: d:\GitRepos\EEG-to-Text_Conformer-Transducer\src\pipelines\training\final_comparison\oov_tokenizer_char.csv

